# Agent Workflow Checklist

This notebook turns the repository contract into a short, executable checklist for agents and contributors.


## 1. Load The Shared Workflow Scaffold

### 1.1 Reuse Tested Notebook Helpers

Notebook cells should orchestrate validations and logging, not own those rules inline.


In [1]:
from pathlib import Path

from repo_rag_lab.notebook_scaffolding import build_agent_workflow_context
from repo_rag_lab.notebook_support import (
    assert_minimum_pass_rate,
    assert_no_validation_issues,
    resolve_repo_root,
    write_notebook_run_log,
)
from repo_rag_lab.utilities import utility_summary
from repo_rag_lab.workflow import ask_repository

root = resolve_repo_root(Path.cwd().resolve())
context = build_agent_workflow_context(root)
print(utility_summary(root))


INFO repo_rag_lab.notebook_scaffolding Built agent workflow context with 8 benchmarks and 1 MCP candidates.


Repository utility surfaces:
- make utility-summary / uv run repo-rag utility-summary: list the supported entrypoints
- make ask / uv run repo-rag ask: answer repository-grounded questions
- make ask-dspy / uv run repo-rag ask --use-dspy: answer with the DSPy runtime path
- make ask-live / uv run repo-rag ask-live: answer with retrieved repo evidence plus a live Azure-backed synthesis step
- make dspy-train / uv run repo-rag dspy-train: compile and persist a DSPy RAG program
- make discover-mcp / uv run repo-rag discover-mcp: inspect repo-local MCP candidates
- make azure-manifest / uv run repo-rag azure-manifest: write Azure deployment metadata
- make azure-openai-probe / uv run repo-rag azure-openai-probe: validate the Azure OpenAI runtime contract
- make azure-inference-probe / uv run repo-rag azure-inference-probe: validate the Azure AI Inference runtime contract
- make files-sync / uv run repo-rag sync-file-summaries: regenerate FILES.md and FILES.csv from the tracked repository f

## 2. Validate Repository Question Answering

### 2.1 Start With A Known Question

Use a repository-specific question before trying open-ended prompts so regressions stay obvious.


In [2]:
baseline = ask_repository("What does this repository research?", root=root)
print(baseline.answer)


Question: What does this repository research?

Evidence:
- /tmp/repo-benchmark-broaden-D0oBUc/README.md: # Repository RAG Lab [![CI](https://github.com/realagiorganization/dspy_rag_in_repo_docs_and_impl1/actions/workflows/ci.yml/badge.svg)](https://github.com/realagiorganization/dspy_rag_in_repo_docs_and_impl1/actions/workflows/ci.yml) [![Publ
- /tmp/repo-benchmark-broaden-D0oBUc/documentation/azure-deployment.md: # Azure Deployment Notes This repository does not fine-tune or deploy a model on its own. Its Azure support is limited to writing a small deployment manifest, validating runtime configuration, and reusing those settings for an optional live
- /tmp/repo-benchmark-broaden-D0oBUc/src/repo_rag_lab/retrieval.py: """Baseline chunking and lexical retrieval utilities for repository text.""" from __future__ import annotations import math import re from dataclasses import dataclass from pathlib import Path from .corpus import RepoDocument TOKEN_RE = re.
- /tmp/repo-benchmark-broaden-D0

## 3. Run Notebook Assertions

### 3.1 Validate Samples, MCP Discovery, And Benchmarks

The scaffold returns validation issues, MCP candidates, and benchmark results in one payload.


In [3]:
training_issues = context["training_validation_issues"]
population_issues = context["population_validation_issues"]
assert_no_validation_issues(training_issues, context="training samples")
assert_no_validation_issues(population_issues, context="population candidates")
assert_minimum_pass_rate(context["benchmark_summary"], minimum_pass_rate=1.0)
{
    "mcp_candidates": context["mcp_candidates"],
    "benchmark_summary": context["benchmark_summary"],
}


{'mcp_candidates': ['pyproject.toml'],
 'benchmark_summary': {'case_count': 8,
  'pass_count': 8,
  'pass_rate': 1.0,
  'fully_covered_case_count': 8,
  'fully_covered_rate': 1.0,
  'average_source_recall': 1.0,
  'average_source_precision': 0.46875,
  'average_reciprocal_rank': 1.0,
  'retrieved_source_hits': {'.codex/skills/file-summary-sync/SKILL.md': 1,
   'AGENTS.md': 2,
   'README.AGENTS.md': 2,
   'README.md': 7,
   'documentation/azure-deployment.md': 2,
   'documentation/inspired/dspy-rag-tutorial.md': 1,
   'documentation/inspired/implementing-rag-with-dspy-technical-guide.md': 1,
   'documentation/mcp-discovery.md': 1,
   'documentation/package-api.md': 3,
   'env.md': 1,
   'publication/README.md': 1,
   'src/repo_rag_lab/retrieval.py': 2,
   'src/repo_rag_lab/training_samples.py': 1,
   'src/repo_rag_lab/utilities.py': 2,
   'utilities/README.md': 5},
  'matched_source_hits': {'AGENTS.md': 1,
   'README.md': 3,
   'documentation/azure-deployment.md': 1,
   'documentation/i

## 4. Write A Notebook Run Log

### 4.1 Persist Scaffold Output For Later Review

Notebook logs land in `artifacts/notebook_logs/` so they do not mix with GitHub Actions logs in `samples/logs/`.


In [4]:
log_path = write_notebook_run_log(root, "02-agent-workflow-checklist", context)
print(log_path.relative_to(root))


artifacts/notebook_logs/20260318T082809Z-02-agent-workflow-checklist.json
